# Exercise 1 — Marvel Universe Social Network Analysis

**Course:** Big Data Analytics 
**Dataset:** Marvel Universe Social Network (Kaggle)  


Team Members:
* Esteban Gerardo Leiva Montenegro (esteban.leiva.001@student.uni.lu)
* Isaac Fabián Palma Medina (isaac.palma.001@student.uni.lu)
* José Valdivia Agüero (jose.valdivia.001@student.uni.lu)

## Tasks
- **(a)** Load graph from `edges.csv`, compare vertices with `nodes.csv`
- **(b)** Connected Components analysis
- **(c)** Degree distribution, Clustering Coefficient, Average Path Length
- **(d)** PageRank — top-25 heroes and top-25 comics, compared to degree ranking
- **(e)** Hero co-occurrence pairs, compared to `hero-edge.csv`




## 0. Environment Setup

Initialize SparkSession with GraphFrames support.  
Make sure the cluster has the `graphframes` package available via `spark.jars.packages`.

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, desc, asc, count, lit,
    monotonically_increasing_id, least, greatest
)
from graphframes import GraphFrame

spark = SparkSession.builder \
    .appName("Marvel Universe Graph Analysis") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.1-s_2.12") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

# Reduce verbosity of Spark logs
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("SparkSession ready.")

## (a) Load and Parse the Graph

## Dataset Description
- `nodes.csv` — `(node, type)`: node name and type (`hero` or `comic`)
- `edges.csv` — `(hero, comic)`: which heroes appear in which comics
- `hero-edge.csv` — pairs of heroes that appear together in the same comics


**Steps:**
1. Load `edges.csv` — columns `(hero, comic)`
2. Extract all distinct vertex IDs from both `hero` and `comic` columns
3. Assign a numeric `id` to each vertex (required by GraphFrames)
4. Build the GraphFrame
5. Load `nodes.csv` and compare both vertex sets


In [ ]:

BASE_DIR = "./data/problem_1/kaggle-marvel-universe"

EDGES_PATH     = BASE_DIR+"/edges.csv"
NODES_PATH     = BASE_DIR+"/nodes.csv"
HERO_EDGE_PATH = BASE_DIR+"/hero-edge.csv"

#  Load edges.csv 
t0 = time.time()

edges_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(EDGES_PATH)

edges_raw.printSchema()
edges_raw.show(5)
print(f"Total edges (hero, comic): {edges_raw.count()}")

In [ ]:
# Both hero names and comic names are nodes in the graph

heroes_df = edges_raw.select(col("hero").alias("node"))
comics_df = edges_raw.select(col("comic").alias("node"))

# Union and deduplicate
all_nodes_from_edges = heroes_df.union(comics_df).distinct()

# Assign a unique numeric id to each node string
# zipWithIndex on RDD guarantees uniqueness
nodes_with_id = all_nodes_from_edges.rdd \
    .zipWithIndex() \
    .map(lambda x: (x[1], x[0][0])) \
    .toDF(["id", "node"])

# Cache — reused many times
nodes_with_id.cache()

print(f"Distinct vertices extracted from edges.csv: {nodes_with_id.count()}")
nodes_with_id.show(5)

In [ ]:
#  Build GraphFrame vertices and edges 

# Vertices: (id, node)
vertices = nodes_with_id

# Edges: remap hero and comic string names to numeric IDs
# src = hero_id, dst = comic_id
edges_with_src = edges_raw.join(
    nodes_with_id.select(col("id").alias("src"), col("node").alias("hero")),
    on="hero"
)
edges_final = edges_with_src.join(
    nodes_with_id.select(col("id").alias("dst"), col("node").alias("comic")),
    on="comic"
).select("src", "dst")

edges_final.cache()
print(f"Edges in GraphFrame: {edges_final.count()}")
edges_final.show(5)

# Build the GraphFrame
g = GraphFrame(vertices, edges_final)
print("GraphFrame created successfully.")

In [ ]:
# ── Compare with nodes.csv 

nodes_csv = spark.read \
    .option("header", "true") \
    .csv(NODES_PATH)

nodes_csv.printSchema()
print(f"Total nodes in nodes.csv: {nodes_csv.count()}")
nodes_csv.groupBy("type").count().show()

# Nodes in edges.csv but NOT in nodes.csv
in_edges_not_in_nodes = all_nodes_from_edges \
    .join(nodes_csv, all_nodes_from_edges.node == nodes_csv.node, "left_anti")

print(f"Nodes in edges.csv but missing from nodes.csv: {in_edges_not_in_nodes.count()}")
in_edges_not_in_nodes.show(10)

# Nodes in nodes.csv but NOT in edges.csv
in_nodes_not_in_edges = nodes_csv \
    .join(all_nodes_from_edges, nodes_csv.node == all_nodes_from_edges.node, "left_anti")

print(f"Nodes in nodes.csv but missing from edges.csv: {in_nodes_not_in_edges.count()}")
in_nodes_not_in_edges.show(10)

t1 = time.time()
print(f"\n⏱ Runtime (a): {t1 - t0:.2f} seconds")

## (b) Connected Components Analysis

**Algorithm recap (Lecture Chapter 6, Slide #19):**  
Each vertex starts with its own ID as state. Iteratively, each vertex sends its state to all neighbors and adopts the **minimum** of all received states. When no state changes, vertices with the same state belong to the same connected component.

**What we expect:**  
Likely one giant component (most heroes are connected through shared comics) and several smaller isolated groups.

In [ ]:
t0 = time.time()

# GraphFrames requires a checkpoint directory for connected components
spark.sparkContext.setCheckpointDir("/tmp/marvel_checkpoints")

cc = g.connectedComponents()

# Count how many distinct components exist
component_counts = cc.groupBy("component").count().orderBy(desc("count"))

total_components = component_counts.count()
print(f"Total number of connected components: {total_components}")
print("\nTop 10 largest components (component_id, size):")
component_counts.show(10)

# Size of the largest component
largest = component_counts.first()
print(f"Largest component contains {largest['count']} vertices")
print(f"Fraction of all vertices in largest component: "
      f"{largest['count'] / vertices.count():.2%}")

t1 = time.time()
print(f"\n⏱ Runtime (b): {t1 - t0:.2f} seconds")

## (c) Degree Distribution, Clustering Coefficient, Average Path Length

**Three structural metrics:**

1. **Degree distribution** — how many connections does each node have? Shows if the graph has hubs (few nodes with very high degree) — typical of real-world networks (power law / long tail).

2. **Average clustering coefficient** — for each node, what fraction of its neighbors are also connected to each other? Range 0–1. High values indicate tight local clusters.

3. **Average path length** — average number of hops between any two nodes. Computed via BFS on a 2% sample (full computation would require ~N²/2 pairs which is computationally prohibitive).

In [ ]:
# ── (c.1) Degree Distribution ──────────────────────────────────────────────────
t0 = time.time()

degrees = g.degrees
degrees.cache()

degree_stats = degrees.select("degree").describe()
print("Degree statistics:")
degree_stats.show()

# Top-10 highest degree nodes (most connected)
top_degree = degrees.join(
    vertices, degrees.id == vertices.id
).select(vertices.node, degrees.degree) \
 .orderBy(desc("degree"))

print("Top 10 most connected nodes:")
top_degree.show(10, truncate=False)

# Distribution: how many nodes have each degree value
degree_distribution = degrees.groupBy("degree").count().orderBy("degree")
print("Degree distribution (first 20 values):")
degree_distribution.show(20)

t1 = time.time()
print(f"⏱ Runtime (c.1 - degree): {t1 - t0:.2f} seconds")

In [ ]:
# ── (c.2) Average Clustering Coefficient ───────────────────────────────────────
# Formula (Lecture Slide #29):
#   C(v) = triangles(v) / (degree(v) * (degree(v) - 1) / 2)
# Average C = sum of all C(v) / number of vertices with degree >= 2

t0 = time.time()

triangle_counts = g.triangleCount()
triangle_counts.cache()

print("Triangle count statistics:")
triangle_counts.select("count").describe().show()

# Join triangle counts with degree to compute local clustering coefficient
tri_degree = triangle_counts.join(
    degrees, triangle_counts.id == degrees.id
).select(
    triangle_counts.id,
    triangle_counts["count"].alias("triangles"),
    degrees.degree
)

# Local clustering coefficient
# Nodes with degree < 2 cannot form triangles — coefficient = 0
from pyspark.sql.functions import when, udf
from pyspark.sql.types import DoubleType

tri_degree = tri_degree.withColumn(
    "max_triangles",
    (col("degree") * (col("degree") - 1) / 2.0)
).withColumn(
    "clustering_coeff",
    when(col("max_triangles") == 0, 0.0)
    .otherwise(col("triangles") / col("max_triangles"))
)

# Average clustering coefficient across all vertices
avg_cc = tri_degree.select("clustering_coeff").groupBy().avg("clustering_coeff").first()[0]
print(f"\nAverage Clustering Coefficient: {avg_cc:.6f}")

t1 = time.time()
print(f"⏱ Runtime (c.2 - clustering): {t1 - t0:.2f} seconds")

In [ ]:
# ── (c.3) Approximate Average Path Length via BFS ──────────────────────────────
# Full computation requires O(N^2) pairs — infeasible for large graphs.
# We sample 2% of vertices as BFS landmarks and approximate the distribution.
#
# GraphFrames provides shortestPaths(landmarks) which implements the same
# Pregel-style BFS described in Lecture Slides #30-35:
#   - Each landmark vertex starts a BFS wave
#   - Each node stores a map {landmark_id -> distance}
#   - Messages propagate +1 hop at each iteration
#   - mergeMaps takes the minimum distance seen so far

t0 = time.time()

# Sample 2% of vertex IDs as landmarks
all_ids = vertices.select("id").rdd.map(lambda r: r[0]).collect()
import random
random.seed(42)
sample_size = max(1, int(len(all_ids) * 0.02))
landmark_ids = random.sample(all_ids, sample_size)
# shortestPaths expects string landmarks
landmark_ids_str = [str(x) for x in landmark_ids]

print(f"Total vertices: {len(all_ids)}")
print(f"Sampled landmarks (2%): {sample_size}")

# Run BFS from all landmarks simultaneously
shortest_paths = g.shortestPaths(landmarks=landmark_ids_str)

# Extract all (landmark, target, distance) triples from the distances map
from pyspark.sql.functions import explode, map_keys, map_values

paths_df = shortest_paths.select(
    col("id").alias("target"),
    explode("distances").alias("landmark", "distance")
).filter(col("distance") > 0)  # exclude self-distances

path_stats = paths_df.select("distance").describe()
print("\nApproximate path length statistics:")
path_stats.show()

# Distribution of path lengths
print("Path length distribution:")
paths_df.groupBy("distance").count().orderBy("distance").show()

avg_path = paths_df.groupBy().avg("distance").first()[0]
print(f"Approximate Average Path Length: {avg_path:.4f}")

t1 = time.time()
print(f"⏱ Runtime (c.3 - path length): {t1 - t0:.2f} seconds")

In [ ]:
# ── (c) Summary — Sparse or Dense? ────────────────────────────────────────────

n_vertices = vertices.count()
n_edges    = edges_final.count()
max_edges  = n_vertices * (n_vertices - 1) / 2  # undirected complete graph
density    = n_edges / max_edges

print("=" * 50)
print("GRAPH STRUCTURAL SUMMARY")
print("=" * 50)
print(f"Vertices:                  {n_vertices:,}")
print(f"Edges:                     {n_edges:,}")
print(f"Graph density:             {density:.8f}")
print(f"Average degree:            (see degree stats above)")
print(f"Avg clustering coefficient:{avg_cc:.6f}")
print(f"Approx avg path length:    {avg_path:.4f}")
print("=" * 50)
print()
print("Interpretation:")
print(f"  Density ≈ {density:.6f} → Very SPARSE graph")
print(f"  High clustering coefficient → Local communities exist")
print(f"  Short avg path length → Small-world network structure")
print("  (Few hubs connect many nodes — power-law degree distribution)")

## (d) PageRank — Top-25 Heroes and Top-25 Comics

**PageRank intuition (Lecture Slide #36):**  
A node is important if important nodes point to it. The random surfer model: with probability `ε` jump to any random node, with probability `(1-ε)` follow an edge. PageRank = fraction of time the surfer spends at each node.

**Why compare to degree?**  
Degree measures quantity of connections. PageRank measures quality — a hero connected to few but very central comics can rank higher than a hero connected to many obscure comics.

In [ ]:
t0 = time.time()

# Run PageRank
# resetProbability = ε = 0.15 (random jump probability)
# maxIter = 10 iterations (sufficient for convergence on most real graphs)
pagerank = g.pageRank(resetProbability=0.15, maxIter=10)

# Join PageRank scores with node names and types
pr_vertices = pagerank.vertices  # has columns: id, pagerank

# Join with original node names
pr_named = pr_vertices.join(
    vertices.select(col("id").alias("v_id"), col("node")),
    pr_vertices.id == col("v_id")
).select("node", "pagerank")

# Join with nodes.csv to get node type (hero / comic)
pr_typed = pr_named.join(
    nodes_csv.select(col("node").alias("n_node"), col("type")),
    pr_named.node == col("n_node"),
    "left"  # some nodes may not exist in nodes.csv (found in (a))
).select("node", "type", "pagerank")

t1 = time.time()
print(f"⏱ PageRank computation: {t1 - t0:.2f} seconds")

In [ ]:
# ── Top-25 Heroes by PageRank ──────────────────────────────────────────────────
print("TOP-25 HEROES BY PAGERANK")
print("=" * 50)
pr_typed.filter(col("type") == "hero") \
        .orderBy(desc("pagerank")) \
        .show(25, truncate=False)

# ── Top-25 Comics by PageRank ──────────────────────────────────────────────────
print("TOP-25 COMICS BY PAGERANK")
print("=" * 50)
pr_typed.filter(col("type") == "comic") \
        .orderBy(desc("pagerank")) \
        .show(25, truncate=False)

In [ ]:
#  Top-25 by Degree (for comparison) 

# Join degrees with node names and types
degree_typed = degrees.join(
    vertices.select(col("id").alias("v_id"), col("node")),
    degrees.id == col("v_id")
).join(
    nodes_csv.select(col("node").alias("n_node"), col("type")),
    col("node") == col("n_node"),
    "left"
).select("node", "type", "degree")

print("TOP-25 HEROES BY DEGREE")
print("=" * 50)
degree_typed.filter(col("type") == "hero") \
            .orderBy(desc("degree")) \
            .show(25, truncate=False)

print("TOP-25 COMICS BY DEGREE")
print("=" * 50)
degree_typed.filter(col("type") == "comic") \
            .orderBy(desc("degree")) \
            .show(25, truncate=False)

In [ ]:
#  Compare PageRank ranking vs Degree ranking 
# Add rank numbers and join both rankings side by side

from pyspark.sql.window import Window
from pyspark.sql.functions import rank as spark_rank

window_pr     = Window.partitionBy("type").orderBy(desc("pagerank"))
window_degree = Window.partitionBy("type").orderBy(desc("degree"))

pr_ranked = pr_typed.withColumn("pr_rank", spark_rank().over(window_pr))
deg_ranked = degree_typed.withColumn("deg_rank", spark_rank().over(window_degree))

# Join on node name for heroes
comparison_heroes = pr_ranked.filter((col("pr_rank") <= 25) & (col("type") == "hero")) \
    .join(
        deg_ranked.filter(col("type") == "hero").select(
            col("node").alias("d_node"), "deg_rank", "degree"
        ),
        pr_ranked.node == col("d_node"),
        "left"
    ).select("node", "pr_rank", col("pagerank").cast("double"), "deg_rank", "degree") \
     .orderBy("pr_rank")

print("HERO COMPARISON — PageRank rank vs Degree rank (top 25 by PR)")
print("A large gap between pr_rank and deg_rank = PageRank captures something degree misses")
comparison_heroes.show(25, truncate=False)

## (e) Hero Co-occurrence Pairs

**Methodology (same as Lecture Chapter 6 — MeSH topic co-occurrences):**  
Two heroes co-occur if they appear in the same comic. We perform a **self-join** on `edges.csv` grouping by comic — for each comic, every pair of heroes that appears in it forms a co-occurrence edge.

The result is then compared with `hero-edge.csv` to find:
- Pairs we computed that are also in `hero-edge.csv` ✅
- Pairs in `hero-edge.csv` not in our computation ❓
- Pairs we computed not in `hero-edge.csv` ❓

In [ ]:
t0 = time.time()

# ── Self-join edges on comic to generate hero pairs ────────────────────────────
# For each comic, find all pairs (heroA, heroB) where heroA < heroB
# The heroA < heroB condition avoids duplicates (A,B) and (B,A) — same as
# the .sorted.combinations(2) used in the Scala slides

hero_pairs = edges_raw.alias("e1").join(
    edges_raw.alias("e2"),
    on=col("e1.comic") == col("e2.comic")  # same comic
).filter(
    col("e1.hero") < col("e2.hero")         # canonical order — avoids (A,B) and (B,A)
).select(
    col("e1.hero").alias("hero1"),
    col("e2.hero").alias("hero2")
).distinct()  # one pair per (hero1, hero2) regardless of how many comics they share

hero_pairs.cache()
computed_count = hero_pairs.count()
print(f"Computed hero co-occurrence pairs: {computed_count:,}")
hero_pairs.show(10, truncate=False)

t1 = time.time()
print(f"⏱ Runtime (e - pair generation): {t1 - t0:.2f} seconds")

In [ ]:
# ── Load hero-edge.csv ─────────────────────────────────────────────────────────
hero_edge_csv = spark.read \
    .option("header", "true") \
    .csv(HERO_EDGE_PATH)

hero_edge_csv.printSchema()
print(f"Pairs in hero-edge.csv: {hero_edge_csv.count():,}")
hero_edge_csv.show(5, truncate=False)

In [ ]:
# ── Normalize hero-edge.csv to canonical order (heroA < heroB) ────────────────
# hero-edge.csv may have pairs in either order — normalize for fair comparison

# Detect column names from hero-edge.csv
c1, c2 = hero_edge_csv.columns[0], hero_edge_csv.columns[1]

hero_edge_normalized = hero_edge_csv.select(
    least(col(c1), col(c2)).alias("hero1"),
    greatest(col(c1), col(c2)).alias("hero2")
).distinct()

print(f"Normalized pairs in hero-edge.csv: {hero_edge_normalized.count():,}")

# ── Comparison ─────────────────────────────────────────────────────────────────

# Pairs we computed that are also in hero-edge.csv
in_both = hero_pairs.join(hero_edge_normalized, on=["hero1", "hero2"])
print(f"\nPairs in BOTH (our computation ∩ hero-edge.csv): {in_both.count():,}")

# Pairs we computed but NOT in hero-edge.csv
only_computed = hero_pairs.join(
    hero_edge_normalized, on=["hero1", "hero2"], how="left_anti"
)
print(f"Pairs only in our computation (not in hero-edge.csv): {only_computed.count():,}")
only_computed.show(10, truncate=False)

# Pairs in hero-edge.csv but NOT in our computation
only_in_file = hero_edge_normalized.join(
    hero_pairs, on=["hero1", "hero2"], how="left_anti"
)
print(f"Pairs only in hero-edge.csv (not in our computation): {only_in_file.count():,}")
only_in_file.show(10, truncate=False)

In [ ]:
# ── (e) Summary ────────────────────────────────────────────────────────────────
print("=" * 55)
print("HERO CO-OCCURRENCE COMPARISON SUMMARY")
print("=" * 55)
print(f"Pairs computed from edges.csv:       {computed_count:,}")
print(f"Pairs in hero-edge.csv:              {hero_edge_normalized.count():,}")
print(f"Pairs in both:                       {in_both.count():,}")
print(f"Only in our computation:             {only_computed.count():,}")
print(f"Only in hero-edge.csv:               {only_in_file.count():,}")
print("=" * 55)
print()
print("Interpretation:")
print("  If 'only in hero-edge.csv' > 0: hero-edge.csv may include")
print("  pairs from a different or extended version of the dataset.")
print("  If 'only in our computation' > 0: we computed more pairs")
print("  possibly because edges.csv is more complete.")

## Final Summary

| Task | Result |
|------|--------|
| **(a)** Vertices from edges.csv vs nodes.csv | See output above — any discrepancy reported |
| **(b)** Connected Components | N components, largest covers X% of vertices |
| **(c)** Degree / Clustering / Path Length | Sparse graph, small-world structure |
| **(d)** PageRank vs Degree ranking | Heroes with fewer but more central connections rank differently |
| **(e)** Hero pairs vs hero-edge.csv | Differences explained by dataset version |

**AI Usage:**  
Claude (claude.ai) was used throughout this exercise.  
Key prompting steps:
- Understanding GraphFrames API vs GraphX Scala API
- Understanding the Pregel BFS algorithm for path length (why +1, what the map stores, when it stops)
- Understanding PageRank convergence and the role of the reset probability ε
- Understanding connected components — minimum state propagation algorithm
- Understanding clustering coefficient formula and edge case for degree < 2

In [ ]:
spark.stop()
print("SparkSession stopped.")